In [16]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

In [17]:
import pandas as pd

train = pd.read_csv("train_FE1.csv")

In [18]:
# Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]
 
# Split internal: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [19]:
# Kolom kategorikal dan numerik

# 1. Definisi Manual untuk Categorical Features
categorical_features = [
    "gender",
    "stress_level",
    "academic_work_impact",
    "sleep_deprived_flag",
    "weekend_binge_flag",
    "screen_risk_bin",
    "sleep_risk_bin",
    "notif_risk_bin",
    "extreme_addiction_pattern",
    "compulsive_usage_flag"
]

# 2. Loop Otomatis untuk Numeric Features
numeric_features = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
    "screen_time_gap",
    "weekend_screen_ratio",
    "social_media_ratio",
    "gaming_ratio",
    "notifications_per_screen_hour",
    "screen_to_sleep_ratio",
    "total_active_screen_hours",
    "non_sleep_screen_share",
    "app_opens_per_screen_hour",
    "notif_to_open_ratio",
    "leisure_screen_hours",
    "leisure_ratio",
    "unaccounted_screen_time",
    "stress_screen_interaction",
    "dominant_activity_ratio",
    "activity_entropy",
    "productive_hours",
    "leisure_vs_productive_gap",
    "leisure_to_productive_ratio",
    "social_vs_gaming_gap",
    "social_to_gaming_ratio",
    "screen_sleep_gap",
    "screen_sleep_product",
    "weekend_binge_gap",
    "notif_sleep_pressure",
    "opens_sleep_pressure",
    "stress_sleep_interaction",
    "stress_leisure_interaction",
    "stress_notif_interaction",
    "opens_per_leisure_hour",
    "notif_per_leisure_hour",
    "notif_open_gap",
    "notif_open_abs_gap",
    "unaccounted_screen_ratio",
    "unaccounted_abs",
    "accounted_screen_hours",
    "accounted_screen_ratio"
] 

# Preprocessing yang sama untuk seluruh model
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

# 10-fold CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [20]:
# Model dan parameter dasar. ROC-AUC = 0.96051
models = {
    "HistGradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}
model = models['HistGradient Boosting']
print(model)


HistGradientBoostingClassifier(l2_regularization=1.0, learning_rate=0.08,
                               max_iter=300, random_state=42)


In [21]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform

In [22]:
# Pipeline baru khusus untuk tuning
tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True
    ))
])

In [23]:
# Rentang parameter yang akan dicoba
param_distributions = {
    # Seberapa besar langkah belajar model
    "model__learning_rate": loguniform(0.02, 0.15),

    # Jumlah maksimal tahap boosting
    "model__max_iter": randint(200, 701),

    # Kompleksitas setiap pohon
    "model__max_leaf_nodes": randint(15, 64),

    # Minimum jumlah data pada daun/node akhir
    "model__min_samples_leaf": randint(10, 51),

    # Regularisasi untuk menahan overfitting
    "model__l2_regularization": loguniform(0.0001, 10.0)
}

In [24]:
# Random search: mencoba kombinasi parameter secara acak
random_search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=12,                 # 12 kombinasi parameter
    scoring="roc_auc",
    cv=cv,                     # StratifiedKFold 10-fold
    n_jobs=2,                  # Naikkan bila RAM/CPU aman
    verbose=3,
    random_state=42,
    refit=True,                # setelah menemukan terbaik, fit ulang pada seluruh X_train
    return_train_score=True 
)

In [25]:
# # Mulai tuning
# random_search.fit(X_train, y_train)

# print("\nParameter terbaik:")
# print(random_search.best_params_)

# print(f"\nROC-AUC CV terbaik: {random_search.best_score_:.5f}")

In [26]:
# # Ambil model dan parameter terbaik
# best_pipeline = random_search.best_estimator_
# best_params = random_search.best_params_

In [27]:
# from sklearn.metrics import roc_auc_score

# y_test_prob = best_pipeline.predict_proba(X_test)[:, 1]

# print("ROC-AUC internal test:",
#       roc_auc_score(y_test, y_test_prob))

In [28]:
# import pandas as pd
# import joblib

# pd.DataFrame(random_search.cv_results_).sort_values(
#     "rank_test_score"
# ).to_csv("random_search_resultsFE1.csv", index=False)

# joblib.dump(best_pipeline, "histgradient4_tunedFE1.joblib")

# import json

# with open("best_paramsFE1.json", "w") as file:
#     json.dump(random_search.best_params_, file, indent=4)

# print("Hasil tuning dan model terbaik berhasil disimpan.")

In [29]:
# import json

# # 1. Load parameter dari file JSON
# with open("best_paramsFE1.json", "r") as f:
#     best_params = json.load(f)
    
# best_params = {
#     key.replace("model__", ""): value for key, value in best_params.items()
# }

# # 2. Gabungkan ke dalam pipeline
# tuning_pipeline = Pipeline(
#     [
#         ("preprocessor", preprocessor),
#         (
#             "model",
#             HistGradientBoostingClassifier(
#                 **best_params,  # Unpack parameter dari JSON di sini
#                 random_state=42,
#                 early_stopping=True
#             ),
#         ),
#     ]
# )

In [30]:
# tuning_pipeline.fit(X, y)

In [31]:
# test = pd.read_csv('test_FE1.csv')

# test_id = test['id']
# Xfinaltest = test.drop(columns=['id'])


In [32]:
# # Probabilitas seseorang masuk kelas addicted_label = 1
# y_prob = tuning_pipeline.predict_proba(Xfinaltest)[:, 1]

# # Label final berdasarkan threshold default 0.5
# y_pred = tuning_pipeline.predict(Xfinaltest)

In [33]:
# submission = pd.DataFrame({
#     "id": test_id,
#     "addicted_label": y_prob
# })

# submission.to_csv("submission4.csv", index=False)

In [34]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint, uniform

In [35]:
new_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True,
        scoring="loss"
    ))
])

In [36]:
param_distributions = {
    "model__learning_rate": loguniform(0.02, 0.20),
    "model__max_iter": randint(250, 1201),
    "model__max_leaf_nodes": randint(8, 80),
    "model__max_depth": [None, 3, 4, 5, 6, 8, 10],
    "model__min_samples_leaf": randint(10, 101),
    "model__l2_regularization": loguniform(1e-3, 50),
    "model__max_features": uniform(0.6, 0.4),
    "model__max_bins": [63, 127, 255],
    "model__n_iter_no_change": randint(10, 60),
    "model__tol": loguniform(1e-8, 1e-4)
}

random_search = RandomizedSearchCV(
    estimator=new_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="roc_auc",
    cv=cv,
    n_jobs=2,
    verbose=2,
    random_state=42,
    refit=True,
    return_train_score=True,
    error_score="raise"
)

In [ ]:
random_search.fit(X_train, y_train)

print("Parameter terbaik:")
print(random_search.best_params_)

print(f"ROC-AUC CV terbaik: {random_search.best_score_:.12f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
